# Autograd

In [76]:
def dy_dx(x):
    return 2*x

print(dy_dx(3))

# but this is not possible for any other function...

6


In [77]:
import torch

In [78]:
x = torch.tensor(3.0, requires_grad= True)

y = x**2

print(y,x)

tensor(9., grad_fn=<PowBackward0>) tensor(3., requires_grad=True)


In [79]:
y.backward()
x.grad

tensor(6.)

In [80]:
z = torch.sin(y)
print(z)

tensor(0.4121, grad_fn=<SinBackward0>)


In [ ]:
z.backward()
x.grad

In [67]:
x = torch.tensor(6.7)
y = torch.tensor(0.0)

w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad= True)

In [68]:
def cross_entropy_loss(prediction, target):
    epsilon = 1e-8 # to prevent log(0)
    prediction = torch.clamp(prediction, epsilon, 1- epsilon)
    return -(target*torch.log(prediction) + (1 - target)*torch.log(1 - prediction))

In [69]:
z = w*x + b
y_pred = torch.sigmoid(z)

loss = cross_entropy_loss(y_pred, y)


In [70]:
loss.backward()

print(w.grad)
print(b.grad)


tensor(6.6918)
tensor(0.9988)


# Working with tensors

In [81]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad= True)

y = (x**2).mean()

y.backward()

print(x.grad)

tensor([0.6667, 1.3333, 2.0000])


# Clearing gradients

- gradients may accumulate which is not desriable

In [ ]:
x.requires_grad = False

# another way is be detaching

z = x.detach()

with torch.no_grad():
    y = x**2

# Pytorch Training Pipeline

- So far we covered introduction to pytorch, history , Tensors, Computation graphs and autograd. 
- Now we will take a dataset and we will creat a neural network. we want to learn the training pipeline of Pytorch.

Plan 
1. We will build a simple neural network
2. Train it on a real world dataset
3. we will mimic PyTorch workflow
4. Will have a lot of manual elements
5. End result is not important we just want to understand how to code the pipeline using pytorch

Code Flow
- Load dataset
- basic Preprocessing
- training proces
    a. Create the model
    b. Forward pass
    c. Loss
    d. Back prop
    e. Parameters update
- Model evaluation




We will be using Breast cancer dataset

'https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv'

In [126]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

df = pd.read_csv("https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv")
df.head()
df

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.380,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.990,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.570,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.910,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.540,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,926424,M,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,1.1760,1.2560,7.673,158.70,0.010300,0.02891,0.05198,0.02454,0.01114,0.004239,25.450,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115,NaN
565,926682,M,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,0.7655,2.4630,5.203,99.04,0.005769,0.02423,0.03950,0.01678,0.01898,0.002498,23.690,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637,NaN
566,926954,M,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,0.4564,1.0750,3.425,48.55,0.005903,0.03731,0.04730,0.01557,0.01318,0.003892,18.980,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820,NaN
567,927241,M,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,0.7260,1.5950,5.772,86.22,0.006522,0.06158,0.07117,0.01664,0.02324,0.006185,25.740,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400,NaN


In [99]:
df.drop(columns=['id', 'Unnamed: 32'], inplace = True)


In [100]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,1:], df.iloc[:, 0], test_size = 0.2)

In [101]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [102]:
X_train

array([[-1.12304627, -0.42598605, -1.10834868, ..., -0.88460142,
         0.23750896,  0.22062449],
       [ 1.60807157,  1.39915064,  1.53185016, ...,  0.51083884,
        -1.00157034, -1.2373103 ],
       [-0.30167927,  0.71991637, -0.39577345, ..., -1.59869512,
        -0.99831816, -1.37092739],
       ...,
       [ 0.48195709, -1.30557   ,  0.44758825, ...,  0.36952843,
        -0.30235236, -0.70930731],
       [-1.04758499, -0.93174682, -1.04506491, ..., -1.05524909,
        -0.4779699 , -0.06385059],
       [ 0.85926349,  0.24103178,  0.77244493, ..., -0.05747475,
         0.59524839, -1.03365201]], shape=(455, 30))

In [ ]:
# Label encoding as out nn cannot understand alphabets

encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

# this is a binary classification problem


In [107]:
# now so far w are were working with numpy array and now we need to convert it into pytorch tensors

X_train_tensor = torch.from_numpy(X_train)
y_train_tensor = torch.from_numpy(y_train)
X_test_tensor = torch.from_numpy(X_test)
y_test_tensor = torch.from_numpy(y_test)


y_train_tensor.shape

torch.Size([455])

In [108]:
X_train_tensor.shape

torch.Size([455, 30])

In [176]:
# Defining model

class MySimpleNN():

    def __init__(self, X):

        # since there are 30 featureswe will have 30 weights and 1 bias since this is a single neuron nn
        self.weights = torch.rand(X.shape[1], 1, dtype=torch.float64, requires_grad= True)
        self.bias = torch.zeros(1, dtype=torch.float64, requires_grad= True)

    def forward(self, X):
        z = torch.matmul(X, self.weights) + self.bias
        y_pred = torch.sigmoid(z)

        return y_pred

    def loss_function(self, y_pred, y):
        # clamp predicction to avoid log(0)
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred, epsilon, 1- epsilon)

        #calculate loss
        return -(y*torch.log(y_pred) + (1 - y)*torch.log(1 - y_pred)).mean()
    

In [177]:
# Important Parameters

learning_rate = 0.1
epochs = 25

In [148]:
# Training Pipeline

#create model
model = MySimpleNN(X_train_tensor)
model.weights

# define loop

for epoch in range(epochs):

    # forward pass
    y_pred = model.forward(X_train_tensor)
    
    # loss calculate
    loss = model.loss_function(y_pred, y_train_tensor)
    print(f"Epoch: {epoch + 1}, Loss: {loss.item()}")

    # backwared pass
    loss.backward()

    # parameters update
    with torch.no_grad():
        model.weights -= learning_rate + model.weights.grad
        model.bias -= learning_rate + model.bias.grad
    # zero gradients
    model.weights.grad.zero_()
    model.bias.grad.zero_()

Epoch: 1, Loss: 3.707866149376292
Epoch: 2, Loss: 1.6284398477625102
Epoch: 3, Loss: 1.348065227037962
Epoch: 4, Loss: 0.7287034182803718
Epoch: 5, Loss: 0.8589158198823073
Epoch: 6, Loss: 0.7205762137891477
Epoch: 7, Loss: 0.8373777727358428
Epoch: 8, Loss: 0.7604375647269618
Epoch: 9, Loss: 0.8220224567970635
Epoch: 10, Loss: 0.797596887414228
Epoch: 11, Loss: 0.8252367172466171
Epoch: 12, Loss: 0.8238799912859193
Epoch: 13, Loss: 0.8371194452982743
Epoch: 14, Loss: 0.8425026639952404
Epoch: 15, Loss: 0.8509267862824742
Epoch: 16, Loss: 0.8574044787447888
Epoch: 17, Loss: 0.8641948361075458
Epoch: 18, Loss: 0.8703919808373481
Epoch: 19, Loss: 0.8764150985585327
Epoch: 20, Loss: 0.8821416228601732
Epoch: 21, Loss: 0.8876570809565978
Epoch: 22, Loss: 0.8929635119944552
Epoch: 23, Loss: 0.89809006713826
Epoch: 24, Loss: 0.9030526487773667
Epoch: 25, Loss: 0.907868762905811


In [149]:
# Evaluation

with torch.no_grad():
    y_pred = model.forward(X_test_tensor)
    y_pred = (y_pred > 0.5).float()
    accuracy = (y_pred == y_test_tensor).float().mean()
    print(f"Accuracy : {accuracy}")

Accuracy : 0.5507848858833313


# PyTorch NN Module

- very importnt module and we are going to use it everywhere.

So far:
4 videos completed... PyTorch fundamentals done. intro, tensor, autograd. and we saw the training pipeline. now we will see how to improve the training module.

There are 2 modules specifically which are really useful
1. The nn module
2. torch.optim module

What improvements in training pipeline?

- we will use nn module to build our neuron
- we will use the built in loss function
- we were using the activation function manually.. we will use nn module instead.
- we were manually updating the paprameters before but now we will use the torch.optim

The nn module

- it is the core library that provides a wide array of classes and fuctions designed to help developers build neural networks efficiently and effectively/
- it abstracts the complexity by offering pre-built layers, loss function, activation functions and other utilities, enabling you to focus on designing and experimenting with the model architecture

Key Components
- Modules(Layers)
    - include layers like nn.linear(fully connected layer)
    - nn.conv2d (Convolution layer)
    - nn.LSTM (recurrent layer) etc
    - nn.Module: The base class for all neural network modules. Your custom models and layers should subclass this class

- Activation Functions
    - Relu, sigmmoid etc

- Loss fucntion
    - crossntropy, MSEloss etc

- Container Modules
    - nn.Sequential: A sequential container to stack layers in order

- Regularization and Dropour
    - Layers like nn.Dropout and nn.BatchNorm2d help prevent over fitting and improve the model's ability to generalize to new data

In [150]:
import torch
import torch.nn as nn

class Model(nn.Module):

    def __init__(self, num_features):

        # Architecture of neural network
        super().__init__()
        self.linear = nn.Linear(num_features, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, features):

        out = self.linear(features)   # wx + b
        out = self.sigmoid(out)

        return out

In [151]:
# create fake dataset
features = torch.rand(10,5)  # 5 features

# create model
model = Model(features.shape[1])

# call forward pass
model(features) # Note: this is the standardway. dont write model.forward(features). This is via the concept of magic methods



tensor([[0.5458],
        [0.5369],
        [0.4274],
        [0.5279],
        [0.4640],
        [0.4516],
        [0.4444],
        [0.4736],
        [0.4579],
        [0.4890]], grad_fn=<SigmoidBackward0>)

In [152]:
model.linear.weight

Parameter containing:
tensor([[ 0.1927,  0.2582,  0.0102, -0.4281,  0.2643]], requires_grad=True)

In [153]:
model.linear.bias

Parameter containing:
tensor([-0.2732], requires_grad=True)

- To visualise this network there is a library called torchinfo

In [163]:
%pip install torchinfo

  Using cached torchinfo-1.8.0-py3-none-any.whl.metadata (21 kB)
Using cached torchinfo-1.8.0-py3-none-any.whl (23 kB)
Note: you may need to restart the kernel to use updated packages.


In [164]:
% pip install --upgrade pip

UsageError: Line magic function `%` not found.


In [167]:
from torchinfo import summary

summary(model, input_size = (10,5))

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [10, 1]                   --
├─Linear: 1-1                            [10, 1]                   6
├─Sigmoid: 1-2                           [10, 1]                   --
Total params: 6
Trainable params: 6
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

In [ ]:
# creating the nn with hidden layer

import torch
import torch.nn as nn

class Model(nn.Module):

    def __init__(self, num_features):

        # Architecture of neural network with hidden layer
        super().__init__()
        self.linear1 = nn.Linear(num_features, 3)
        self.ReLU = nn.ReLU()
        self.linear2 = nn.Linear(3,1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, features):

        out = self.linear1(features)  # wx + b
        out = self.ReLU(out)
        out = self.linear2(out)  
        out = self.sigmoid(out)

        # this can be further simplifies using sequential containers

        return out

In [169]:
# create fake dataset
features = torch.rand(10,5)  # 5 features

# create model
model = Model(features.shape[1])

# call forward pass
model(features) # Note: this is the standardway. dont write model.forward(features). This is via the concept of magic methods



tensor([[0.5271],
        [0.4923],
        [0.5000],
        [0.5158],
        [0.5245],
        [0.5142],
        [0.5069],
        [0.5036],
        [0.5149],
        [0.5006]], grad_fn=<SigmoidBackward0>)

In [171]:
model.linear1.weight


Parameter containing:
tensor([[-0.3391,  0.2352, -0.3765, -0.2146, -0.3940],
        [ 0.0453, -0.1572,  0.1286,  0.1513,  0.2519],
        [ 0.1241, -0.1736, -0.0143,  0.1165, -0.2429]], requires_grad=True)

In [172]:
summary(model, input_size= (10,5))

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [10, 1]                   --
├─Linear: 1-1                            [10, 3]                   18
├─ReLU: 1-2                              [10, 3]                   --
├─Linear: 1-3                            [10, 1]                   4
├─Sigmoid: 1-4                           [10, 1]                   --
Total params: 22
Trainable params: 22
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

In [ ]:
# simplifying forward pass using Sequential container

import torch
import torch.nn as nn

class Model(nn.Module):

    def __init__(self, num_features):

        # Architecture of neural network with hidden layer
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(num_features, 3),
            nn.ReLU(),
            nn.Linear(3,1),
            nn.Sigmoid()
        )

    def forward(self, features):

        out = self.network(features)

        # this can be further simplifies using sequential containers

        return out

Torch.optim module

torch.optim is a module that provides a variety of optimisation algorithms used to update the paramteres of your model during training.

It includes common optimizers like Stochastic Gradient Descent, Adam, RMSprop and more

It handles weihts updates efficiently, including additional features like learning rate schedulng and weight decay (regularization)

The  model.parameters() method Pytoch retrievs an iterator over all the trainable parameters (weights and biases) in a model . There parameters are instances of torch.nn.Parameters and includeL weigt and biases

The optimizer uses these parameters to compute gradient and update them during training.

In [ ]:
# define optimizer
model = MySimpleNN()
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)
# define loop

for epoch in range(epochs):

    # forward pass
    y_pred = model.forward(X_train_tensor)
    
    # loss calculate
    loss = model.loss_function(y_pred, y_train_tensor)
    print(f"Epoch: {epoch + 1}, Loss: {loss.item()}")

    optimizer.zero_grad()
    # backwared pass
    loss.backward()

    # parameters update
    # with torch.no_grad():
    #     model.weights -= learning_rate + model.weights.grad
    #     model.bias -= learning_rate + model.bias.grad

    optimizer.step()

    # # zero gradients
    # model.weights.grad.zero_()
    # model.bias.grad.zero_()

    # optimizer.zero_grad()   ---> you should do this before the backward pass ... it is a good practice


    # this code will not work because it is just an example na